In [1]:
import gmsh
import sys

def make_box_mesh(
    Lx=40.0, Ly=10.0, Lz=6.,
    lc_min=0.3, lc_max=.7,
    dist_min=0.01, dist_max=3.0,
    filename="box_centerline_top_refine.msh",
    eps_frac=1e-6
):
    # --- reset state for Jupyter reruns ---
    if gmsh.isInitialized():
        gmsh.clear()          # clears current model/entities
    else:
        gmsh.initialize(sys.argv)

    gmsh.option.setNumber("General.Terminal", 1)  # show gmsh messages in notebook
    gmsh.model.add("box_centerline_top_refine")

    eps = eps_frac * Lx  # keep curve off boundary edges

    # --- geometry (OCC) ---
    vol = gmsh.model.occ.addBox(0.0, 0.0, 0.0, Lx, Ly, Lz)

    # centerline on top face (along x at y=Ly/2, z=Lz), offset from edges
    pA = gmsh.model.occ.addPoint(eps,      Ly/2.0, Lz, lc_min)
    pB = gmsh.model.occ.addPoint(Lx - eps, Ly/2.0, Lz, lc_min)
    c_top = gmsh.model.occ.addLine(pA, pB)

    gmsh.model.occ.synchronize()

    # --- robustly find the top face (largest center-of-mass z among boundary faces) ---
    bnd = gmsh.model.getBoundary([(3, vol)], oriented=False, recursive=False)
    faces = [tag for (dim, tag) in bnd if dim == 2]
    if not faces:
        raise RuntimeError("No boundary faces found for the volume.")

    top_face = max(faces, key=lambda s: gmsh.model.occ.getCenterOfMass(2, s)[2])

    # --- embed curve so mesh conforms to it ---
    gmsh.model.mesh.embed(1, [c_top], 2, top_face)  # curve in top surface
    gmsh.model.mesh.embed(1, [c_top], 3, vol)       # curve in volume

    # --- size field: Distance -> Threshold ---
    f_dist = gmsh.model.mesh.field.add("Distance")
    gmsh.model.mesh.field.setNumbers(f_dist, "CurvesList", [c_top])
    gmsh.model.mesh.field.setNumber(f_dist, "NumPointsPerCurve", 200)

    f_th = gmsh.model.mesh.field.add("Threshold")
    gmsh.model.mesh.field.setNumber(f_th, "InField", f_dist)
    gmsh.model.mesh.field.setNumber(f_th, "SizeMin", lc_min)
    gmsh.model.mesh.field.setNumber(f_th, "SizeMax", lc_max)
    gmsh.model.mesh.field.setNumber(f_th, "DistMin", dist_min)
    gmsh.model.mesh.field.setNumber(f_th, "DistMax", dist_max)

    gmsh.model.mesh.field.setAsBackgroundMesh(f_th)

    # force the background field to control sizes
    gmsh.option.setNumber("Mesh.CharacteristicLengthFromPoints", 0)
    gmsh.option.setNumber("Mesh.CharacteristicLengthFromCurvature", 0)
    gmsh.option.setNumber("Mesh.CharacteristicLengthExtendFromBoundary", 0)
    gmsh.option.setNumber("Mesh.CharacteristicLengthMin", lc_min)
    gmsh.option.setNumber("Mesh.CharacteristicLengthMax", lc_max)

    # --- physical groups (optional but useful) ---
    gmsh.model.addPhysicalGroup(3, [vol], 1)
    gmsh.model.setPhysicalName(3, 1, "box")

    gmsh.model.addPhysicalGroup(2, [top_face], 2)
    gmsh.model.setPhysicalName(2, 2, "top")

    gmsh.model.addPhysicalGroup(1, [c_top], 3)
    gmsh.model.setPhysicalName(1, 3, "top_centerline")

    # --- mesh + write ---
    gmsh.option.setNumber("Mesh.ElementOrder", 1)  # request 2nd order

    gmsh.model.mesh.generate(3)
    gmsh.write(filename)

    # return some quick stats
    nn = gmsh.model.mesh.getNodes()[0].size
    ne_tet = len(gmsh.model.mesh.getElementsByType(4)[1])  # type 4 = tetra4
    return {"file": filename, "top_face": top_face, "curve": c_top, "nodes": nn, "tet4": ne_tet}


In [2]:
make_box_mesh(filename="box_centerline_top_refine.msh")

{'file': 'box_centerline_top_refine.msh',
 'top_face': 6,
 'curve': 13,
 'nodes': 9440,
 'tet4': 178512}

In [3]:
from skfem import MeshTet, Basis, ElementTetP2, FacetBasis

In [4]:
mesh = MeshTet.load("box_centerline_top_refine.msh")

In [5]:
mesh

<skfem MeshTet1 object>
  Number of elements: 44628
  Number of vertices: 9440
  Number of nodes: 9440
  Named subdomains [# elements]: box [44628], gmsh:bounding_entities [6]
  Named boundaries [# facets]: top [3700]

In [6]:
# element and basis
element = ElementTetP2()
basis   = Basis(mesh, element)

In [8]:
basis.mesh.save('non_uni_mesh.vtu')

In [7]:
basis

<skfem CellBasis(MeshTet1, ElementTetP2) object>
  Number of elements: 44628
  Number of DOFs: 67872
  Size: 157090560 B